In [1]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

In [2]:
torch.random.manual_seed(42)
data = torch.empty(20, 2).uniform_(-1, 1)
labels = (data[:, 1] >= data[:, 0]).float().unsqueeze(1)
dataset =TensorDataset(data, labels)
subset = torch.utils.data.Subset(dataset, [2, 3, 4, 5])
loader = DataLoader(subset, batch_size=4)

In [4]:
#_tmp = nn.Linear(2, 1)  # 用同樣的 seed 初始化
#W = _tmp.weight.detach().clone().requires_grad_(True)
#b = _tmp.bias.detach().clone().requires_grad_(True)
#criterion = nn.BCEWithLogitsLoss()
#optimizer = torch.optim.SGD([W, b], lr=0.1)
from itertools import combinations
all_subsets = list(combinations(range(100), 4))
print(len(all_subsets))

3921225


In [ ]:
W = torch.randn(1, 2, requires_grad=True) #Weight [Output=1, Input=2]
b = torch.randn(1, requires_grad=True) #Bias [Output=1, Input=1]
#x @ W.T + b
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.SGD([W, b], lr=0.1)


In [ ]:
# Training loop
losses = []
for epoch in range(100):
    for x, y in loader:
        optimizer.zero_grad()
        loss = criterion((x @ W.T + b), y)
        loss.backward()
        optimizer.step()
    losses.append(loss.item())
print(f'Fianl Loss: {loss.item():.4f}')


Fianl Loss: 0.3075


In [ ]:
#Test set
test_data = torch.cartesian_prod(torch.linspace(-1, 1, 20), torch.linspace(-1, 1, 20))
test_labels = (test_data[:, 1] >= test_data[:, 0]).float().unsqueeze(1)
test_dataset = TensorDataset(test_data, test_labels)
test_loader = DataLoader(test_dataset, batch_size=400, shuffle=False)


In [ ]:
# Evalute testset
with torch.no_grad():
    for x, y in test_loader:
        logits = (x @ W.T + b)
        preds = (logits >= 0).float()
        accuracy = (preds == y).float().mean()
print(f'Test Accuracy: {accuracy.item():.4f}')

Test Accuracy: 0.8950


In [ ]:
#Visualization
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

correct = (preds == y).squeeze().numpy()
x_np = x.numpy()

fig, (ax_loss, ax_scatter) = plt.subplots(1, 2, figsize=(12, 5))

ax_loss.plot(losses)
ax_loss.set_xlabel('Epoch')
ax_loss.set_ylabel('Loss')
ax_loss.set_title('Training Loss')

ax_scatter.scatter(x_np[correct, 0], x_np[correct, 1], color='green', marker='o', label='Correct')
ax_scatter.scatter(x_np[~correct, 0], x_np[~correct, 1], color='red', marker='x', label='Incorrect')
ax_scatter.plot([-1, 1], [-1, 1], color='gray', linestyle='--', label='Decision Boundary (x0=x1)')
ax_scatter.set_xlabel('x0')
ax_scatter.set_ylabel('x1')
ax_scatter.set_title('Testset classification results')
ax_scatter.legend()

plt.tight_layout()
plt.savefig('saved')
print('saved')

saved
